### Chunking and Embedding

In [56]:
from langchain_community.document_loaders import PyMuPDFLoader

loader = PyMuPDFLoader("../data/DBMS_Full_Notes.pdf")
documents = loader.load()
print(len(documents))

49


#### Chunking

In [57]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)
# chunks

for i, chunk in enumerate(chunks[:3]):
    print(f"Chunk {i}")
    print(chunk.page_content)
    print("-" * 50)

Chunk 0
LEC-1: Introduction to DBMS 
1.
What is Data?
a.
Data is a collection of raw, unorganized facts and details like text, observations, figures, symbols,
and descriptions of things etc.
In other words, data does not carry any specific purpose and has no significance by itself.
Moreover, data is measured in terms of bits and bytes – which are basic units of information in the
context of computer storage and processing.
b.
Data can be recorded and doesn’t have any meaning unless processed.
2.
--------------------------------------------------
Chunk 1
b.
Data can be recorded and doesn’t have any meaning unless processed.
2.
Types of Data
a.
Quantitative
i.
Numerical form
ii.
Weight, volume, cost of an item.
b.
Qualitative
i.
Descriptive, but not numerical.
ii.
Name, gender, hair color of a person.
3.
What is Information?
a.
Info. Is processed, organized, and structured data.
b.
It provides context of the data and enables decision making.
c.
Processed data that make sense to us.
d.
--

#### Embedding

In [58]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

embedding = model.encode(chunks[0].page_content)

print(type(embedding))
print(len(embedding))
print(embedding[:10])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4120.93it/s]


<class 'numpy.ndarray'>
384
[ 0.0055649   0.0293134  -0.03320569  0.02676382  0.01399536 -0.04103997
  0.04913381  0.02029777 -0.01778893  0.05071862]


In [59]:
embeddings = model.encode([chunk.page_content for chunk in chunks])

print(embeddings.shape)

(226, 384)


In [ ]:
query = """Why did developers start preferring NoSQL databases?"""

embedded_query = model.encode(query)

embedded_query

In [34]:
from sklearn.metrics.pairwise import cosine_similarity

similarities = cosine_similarity(
    [embedded_query],
    embeddings
)

print(similarities.max())

0.76946986


In [ ]:
best_idx = similarities[0].argmax()

# best_idx
print(chunks[best_idx].page_content)

In [ ]:
import numpy as np

top_k = 5

indices = np.argsort(similarities[0])[-top_k:][::-1]

for idx in indices:
    print(f"Score: {similarities[0][idx]:.4f}")
    print(chunks[idx].page_content[:300])
    print("=" * 80)

### FAISS

In [60]:
import numpy as np

embadding = np.array(embeddings).astype("float32")

print(embeddings.shape)
print(embeddings.dtype)

(226, 384)
float32


In [68]:
import faiss

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)

print(embedding.shape)

(384,)


In [69]:
index.add(embeddings)
print(index.ntotal)

226


In [63]:
query = "Why did developers start preferring NoSQL databases?"

query_embedding = model.encode(query)

query_embedding = np.array(
    [query_embedding]
).astype("float32")

print(query_embedding.shape)

(1, 384)


In [64]:
k = 5

distances, indices = index.search(
    query_embedding,
    k
)

print(distances)
print(indices)

[[0.4610603  0.5274227  0.6514531  0.66151744 0.69928885]]
[[148 147 149 168 177]]


In [65]:
for idx in indices[0]:
    print(f"Chunk {idx}")
    print(chunks[idx].page_content)
    print("=" * 100)

Chunk 148
becoming the primary cost of software development, so NoSQL databases optimised for developer productivity.
2.
Data becoming unstructured more, hence structuring (defining schema in advance) them had becoming costly.
3.
NoSQL databases allow developers to store huge amounts of unstructured data, giving them a lot of flexibility.
4.
Recognising the need to rapidly adapt to changing requirements in a software system. Developers needed the ability to iterate
Chunk 147
3.
Can handle huge amount of data (big data).
4.
Most of the NoSQL are open sources and has the capability of horizontal scaling.
5.
It just stores data in some format other than relational.
2.
History behind NoSQL
1.
NoSQL databases emerged in the late 2000s as the cost of storage dramatically decreased. Gone were the days of needing to 
create a complex, difficult-to-manage data model in order to avoid data duplication. Developers (rather than storage) were
Chunk 149
quickly and make changes throughout their soft

In [71]:
print(index.ntotal)
# print(loaded_index.ntotal)

226


#### Saving the index 

In [72]:
faiss.write_index(
    index,
    "../store/nosql_index.faiss"
)

import pickle

with open("../store/chunks.pkl", "wb") as f:
    pickle.dump(chunks, f)